# Previsão de atrasos de voos com Spark ML (US DOT Flight Delays 2015)

**Disciplina:** Processamento de Dados Massivos — IESB
**Tipo de problema (sorteado):** Classificação
**Ferramentas:** PySpark (DataFrame API + `pyspark.ml`), Python 3.12, Java 17

Este notebook cobre o pipeline completo: ingestão, limpeza, engenharia de atributos, modelagem com Spark ML, avaliação e persistência do modelo.

## 1. Contextualização

### 1.1 A base de dados
| Item | Descrição |
|---|---|
| **Fonte** | Kaggle — [US DOT Flight Delays](https://www.kaggle.com/datasets/usdot/flight-delays) (`usdot/flight-delays`) |
| **Origem dos dados** | U.S. Department of Transportation (DOT), Bureau of Transportation Statistics (BTS), *Airline On-Time Performance* |
| **Período** | Ano de 2015 (janeiro a dezembro) |
| **Arquivo principal** | `flights.csv`: cerca de 5,8 milhões de voos × 31 colunas (592 MB em texto) |
| **Arquivos auxiliares** | `airlines.csv` (códigos e nomes de companhias) e `airports.csv` (aeroportos, cidade, estado, latitude e longitude) |
| **Amostragem** | Nenhuma. Usamos todos os registros do arquivo publicado. Pela descrição do dataset, ele cobre voos domésticos das grandes companhias que reportam ao BTS, e não todos os voos dos EUA. |

A contagem exata de linhas e colunas é calculada com Spark na etapa 2.

### 1.2 Pergunta de negócio
> **Com as informações disponíveis antes da decolagem, um voo vai chegar ao destino com 15 minutos ou mais de atraso?**

Esse limiar de 15 minutos é a convenção usada pelo DOT/FAA para classificar um voo como atrasado. A variável-alvo será binária: `ATRASADO = 1` se `ARRIVAL_DELAY >= 15`, senão `0`.

**Por que isso importa:** companhias aéreas podem realocar equipe e aeronaves; aeroportos podem reorganizar portões; e passageiros podem decidir conexões com mais segurança. Prever o atraso com antecedência reduz custos operacionais e aumenta a satisfação do cliente.

### 1.3 Por que este é um problema de dados massivos
Os "Vs" de big data aplicados a este cenário:

- **Volume:** ~5,8 milhões de registros × 31 colunas em 592 MB de texto. Na memória, um DataFrame desse tamanho ocupa várias vezes o tamanho do arquivo, o que já pressiona ferramentas de máquina única como o pandas.
- **Velocidade:** os dados são gerados continuamente. São cerca de 16 mil voos por dia, e o BTS publica novos meses periodicamente. Um modelo real precisa ser reprocessado e reavaliado com frequência.
- **Variedade:** há atributos numéricos, horários codificados como `HHMM`, categóricas de alta cardinalidade (companhia, ~300 aeroportos de origem e destino) e tabelas auxiliares que exigem *joins*.
- **Veracidade:** muitos campos são nulos por natureza (voos cancelados não têm horário de chegada) e há códigos inconsistentes de aeroporto, o que exige limpeza em escala.

**Por que processamento distribuído:** o Spark divide os dados em partições processadas em paralelo, executa transformações de forma *preguiçosa* (lazy evaluation), otimizando o plano de execução antes de ler os dados, e oferece algoritmos de ML (`pyspark.ml`) que treinam sobre partições, como as árvores do Random Forest. O mesmo código roda de um notebook local até um cluster com centenas de nós.

**Honestidade sobre a escala:** o arquivo de 2015 cabe em um computador pessoal, e neste trabalho o Spark roda em modo local (`local[*]`, usando todos os núcleos da máquina). O argumento de dados massivos está em que o pipeline foi construído para escalar: a série histórica do BTS cobre décadas, e o mesmo notebook processaria vários anos de dados, algo que não caberia na memória de uma única máquina, simplesmente apontando para um cluster, sem reescrever o código.

### 1.4 Limitações já conhecidas
- Dados de um único ano (2015), então a sazonalidade e eventos pontuais podem não generalizar.
- Não há informação de clima nem de congestionamento do tráfego aéreo, que são causas importantes de atraso.
- Atrasos se propagam em cascata (a aeronave que chega atrasada atrasa o próximo voo), e o modelo não enxerga isso diretamente.

## 2. Ingestão

**Conceitos desta etapa**
- **`SparkSession`:** ponto de entrada do Spark. Aqui usamos `local[*]`, um único processo com um *worker* por núcleo.
- **Leitura preguiçosa (lazy):** `spark.read` só planeja a leitura; os dados só são lidos quando uma *ação* (`count`, `show`, `write`) é chamada.
- **Schema manual (`StructType`):** com `inferSchema`, o Spark precisaria de uma passada extra pelos 592 MB só para adivinhar os tipos, e poderia errar (por exemplo, colunas quase vazias). Declarar o schema é mais rápido e deixa os tipos sob nosso controle.
- **Parquet:** formato colunar e comprimido. Depois da primeira leitura do CSV, gravamos Parquet e passamos a ler dele, o que torna as próximas etapas bem mais leves.

In [ ]:
import os
import sys
from pathlib import Path

# Os workers do PySpark chamam "python" pelo nome; no Windows isso pode cair no
# atalho da Microsoft Store. Apontamos explicitamente para o interpretador atual.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# No Windows, o Spark precisa do winutils.exe/hadoop.dll (Hadoop 3.3.x) para gravar
# arquivos (Parquet e model.save()). Definimos o HADOOP_HOME só dentro deste processo.
if sys.platform == "win32":
    hadoop_home = Path(os.environ.get("HADOOP_HOME", Path.home() / "hadoop"))
    if (hadoop_home / "bin" / "winutils.exe").exists():
        os.environ["HADOOP_HOME"] = str(hadoop_home)
        os.environ["PATH"] = str(hadoop_home / "bin") + os.pathsep + os.environ["PATH"]
    else:
        print(f"AVISO: winutils.exe não encontrado em {hadoop_home}\\bin. "
              "A gravação de arquivos pode falhar.")

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")                          # todos os núcleos da máquina
    .appName("flight-delays-classificacao")
    .config("spark.driver.memory", "6g")         # em modo local, o driver executa tudo
    .config("spark.sql.shuffle.partitions", "24")  # padrão é 200; exagerado para uma máquina só
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| núcleos disponíveis:", spark.sparkContext.defaultParallelism)

### 2.1 Download da base a partir da fonte original
Os dados **não** ficam no GitHub: o `kagglehub` baixa a base do Kaggle (com cache local).
As credenciais devem estar nas variáveis de ambiente `KAGGLE_USERNAME` e `KAGGLE_KEY`, ou em `KAGGLE_API_TOKEN` (veja o `README.md`).

In [ ]:
import kagglehub

caminho_kaggle = Path(kagglehub.dataset_download("usdot/flight-delays"))
print("Arquivos baixados em:", caminho_kaggle)
for arquivo in sorted(caminho_kaggle.glob("*.csv")):
    print(f"  {arquivo.name:15s} {arquivo.stat().st_size / 1e6:8.1f} MB")

### 2.2 Schema manual do `flights.csv`
Todas as 31 colunas são declaradas com tipo explícito. Observações:
- Horários (`SCHEDULED_DEPARTURE`, `DEPARTURE_TIME` etc.) vêm como `HHMM` (por exemplo `0005`). Lemos como inteiro (`5` = 00:05) e converteremos para hora e minuto na engenharia de atributos.
- `ORIGIN_AIRPORT` e `DESTINATION_AIRPORT` ficam como texto, pois alguns registros podem trazer códigos numéricos em vez do código IATA de 3 letras. Isso será tratado na limpeza.
- As colunas de atraso por causa (`*_DELAY`) são quase sempre vazias; são nulas, e não zero.

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema_voos = StructType([
    StructField("YEAR",                 IntegerType(), True),
    StructField("MONTH",                IntegerType(), True),
    StructField("DAY",                  IntegerType(), True),
    StructField("DAY_OF_WEEK",          IntegerType(), True),
    StructField("AIRLINE",              StringType(),  True),
    StructField("FLIGHT_NUMBER",        IntegerType(), True),
    StructField("TAIL_NUMBER",          StringType(),  True),
    StructField("ORIGIN_AIRPORT",       StringType(),  True),
    StructField("DESTINATION_AIRPORT",  StringType(),  True),
    StructField("SCHEDULED_DEPARTURE",  IntegerType(), True),
    StructField("DEPARTURE_TIME",       IntegerType(), True),
    StructField("DEPARTURE_DELAY",      IntegerType(), True),
    StructField("TAXI_OUT",             IntegerType(), True),
    StructField("WHEELS_OFF",           IntegerType(), True),
    StructField("SCHEDULED_TIME",       IntegerType(), True),
    StructField("ELAPSED_TIME",         IntegerType(), True),
    StructField("AIR_TIME",             IntegerType(), True),
    StructField("DISTANCE",             IntegerType(), True),
    StructField("WHEELS_ON",            IntegerType(), True),
    StructField("TAXI_IN",              IntegerType(), True),
    StructField("SCHEDULED_ARRIVAL",    IntegerType(), True),
    StructField("ARRIVAL_TIME",         IntegerType(), True),
    StructField("ARRIVAL_DELAY",        IntegerType(), True),
    StructField("DIVERTED",             IntegerType(), True),
    StructField("CANCELLED",            IntegerType(), True),
    StructField("CANCELLATION_REASON",  StringType(),  True),
    StructField("AIR_SYSTEM_DELAY",     IntegerType(), True),
    StructField("SECURITY_DELAY",       IntegerType(), True),
    StructField("AIRLINE_DELAY",        IntegerType(), True),
    StructField("LATE_AIRCRAFT_DELAY",  IntegerType(), True),
    StructField("WEATHER_DELAY",        IntegerType(), True),
])
print("Colunas declaradas:", len(schema_voos.fields))

### 2.3 Leitura do CSV e conversão para Parquet
Na primeira execução lemos o CSV com o schema manual e `mode="FAILFAST"`: se alguma célula não couber no tipo declarado, o Spark **interrompe com erro** em vez de transformá-la silenciosamente em nulo. Em seguida gravamos em Parquet (pasta `data/`, ignorada pelo git) e, daqui em diante, lemos sempre do Parquet.

In [ ]:
DIR_DADOS = Path("data")
DIR_DADOS.mkdir(exist_ok=True)
caminho_parquet = DIR_DADOS / "flights.parquet"

if not caminho_parquet.exists():
    voos_csv = (
        spark.read
        .option("header", True)
        .option("mode", "FAILFAST")          # falha se algum valor violar o schema
        .schema(schema_voos)
        .csv(str(caminho_kaggle / "flights.csv"))
    )
    print("Partições de leitura do CSV:", voos_csv.rdd.getNumPartitions())
    voos_csv.write.mode("overwrite").parquet(str(caminho_parquet))
    print("Parquet gravado em", caminho_parquet)
else:
    print("Parquet já existe, reaproveitando:", caminho_parquet)

voos = spark.read.parquet(str(caminho_parquet))

### 2.4 Tabelas auxiliares
`airlines.csv` e `airports.csv` são minúsculos (dezenas e centenas de linhas), então aqui **usamos `inferSchema`**: o custo de uma passada extra é desprezível, e evita escrever um schema que não traz benefício.

In [ ]:
companhias = (spark.read.option("header", True).option("inferSchema", True)
              .csv(str(caminho_kaggle / "airlines.csv")))
aeroportos = (spark.read.option("header", True).option("inferSchema", True)
              .csv(str(caminho_kaggle / "airports.csv")))
companhias.printSchema()
aeroportos.printSchema()

### 2.5 Verificação da ingestão
Conferimos o tamanho real da base (e o requisito de pelo menos 200 mil registros) e olhamos uma amostra.

In [ ]:
n_linhas = voos.count()          # primeira ação: aqui o Spark de fato lê os dados
n_colunas = len(voos.columns)
print(f"Registros: {n_linhas:,}".replace(",", "."))
print(f"Colunas:   {n_colunas}")
assert n_linhas >= 200_000, "A base precisa ter pelo menos 200 mil registros"
print("Partições do DataFrame:", voos.rdd.getNumPartitions())

# Comparação de armazenamento: CSV (texto) vs Parquet (colunar e comprimido)
tam_csv = (caminho_kaggle / "flights.csv").stat().st_size / 1e6
tam_parquet = sum(p.stat().st_size for p in caminho_parquet.rglob("*") if p.is_file()) / 1e6
print(f"CSV: {tam_csv:.0f} MB | Parquet: {tam_parquet:.0f} MB")

In [ ]:
voos.printSchema()
# Formato vertical: com 31 colunas, a tabela horizontal fica ilegível
voos.show(2, vertical=True, truncate=False)